In [0]:
df = spark.read.format("delta").load(
    "/Volumes/workspace/default/raw_datasets/cleaned_data"
)

In [0]:
from pyspark.sql import functions as F
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
 
print("="*45)
print("  PHASE 2: EDA + STATISTICAL TESTING")
print("="*45)
print(f"Rows in dataset: {df.count():,}")

In [0]:
total   = df.count()
churned = df.filter(F.col("label") == 1).count()
 
print("\nRETENTION OVERVIEW")
print("-"*35)
print(f"Total customers  : {total:,}")
print(f"Churned          : {churned:,}  ({churned/total*100:.1f}%)")
print(f"Retained         : {total-churned:,}  ({(total-churned)/total*100:.1f}%)")
print("\nKey note: 27% churn rate = imbalanced dataset")
print("Use AUC-ROC not just accuracy for model evaluation")
 

In [0]:
df.createOrReplaceTempView("churn_data")
print("\nChurn rate by Contract type:")
display(spark.sql("""
    SELECT Contract,
           COUNT(*)                        AS total_customers,
           SUM(label)                      AS churned,
           ROUND(AVG(label) * 100, 1)      AS churn_rate_pct
    FROM churn_data
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
"""))

In [0]:
print("Churn rate by Internet Service:")
display(spark.sql("""
    SELECT InternetService,
           COUNT(*)                     AS customers,
           ROUND(AVG(label) * 100, 1)   AS churn_rate_pct
    FROM churn_data
    GROUP BY InternetService
    ORDER BY churn_rate_pct DESC
"""))

In [0]:
print("Churn rate by Payment Method:")
display(spark.sql("""
    SELECT PaymentMethod,
           COUNT(*)                     AS customers,
           ROUND(AVG(label) * 100, 1)   AS churn_rate_pct
    FROM churn_data
    GROUP BY PaymentMethod
    ORDER BY churn_rate_pct DESC
"""))

In [0]:
print("Average values: Churned vs Retained:")
display(
    df.groupBy("label").agg(
        F.round(F.avg("tenure"),         1).alias("avg_tenure_months"),
        F.round(F.avg("MonthlyCharges"), 2).alias("avg_monthly_charges"),
        F.round(F.avg("TotalCharges"),   2).alias("avg_total_charges")
    ).orderBy("label")
)

In [0]:
pdf = df.toPandas()  # convert to pandas for scipy stats
 
categorical_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "TechSupport", "StreamingTV",
    "Contract", "PaperlessBilling", "PaymentMethod"
]
 
print("="*60)
print("  CHI-SQUARE TEST RESULTS")
print("  H0: Feature is independent of churn")
print("  p < 0.05 → reject H0 → feature IS related to churn")
print("="*60)
print(f"\n{'Feature':<22} {'Chi2':>8} {'p-value':>10}  Result")
print("-"*58)
 
significant_cats = []
for col_name in categorical_cols:
    contingency            = pd.crosstab(pdf[col_name], pdf["Churn"])
    chi2, p, dof, _        = stats.chi2_contingency(contingency)
    result                 = "SIGNIFICANT ✓" if p < 0.05 else "not significant"
    if p < 0.05:
        significant_cats.append(col_name)
    print(f"{col_name:<22} {chi2:>8.2f} {p:>10.4f}  {result}")
 
print(f"\n→ {len(significant_cats)} significant features: {significant_cats}")

In [0]:
churned_pdf  = pdf[pdf["Churn"] == "Yes"]
retained_pdf = pdf[pdf["Churn"] == "No"]
 
print("\n" + "="*70)
print("  T-TEST RESULTS")
print("  H0: No difference in mean between churned vs retained")
print("  p < 0.05 → reject H0 → means ARE significantly different")
print("="*70)
print(f"\n{'Feature':<20} {'Churned':>12} {'Retained':>12} {'p-value':>10}  Result")
print("-"*68)
 
for col_name in ["tenure", "MonthlyCharges", "TotalCharges"]:
    t, p = stats.ttest_ind(
        churned_pdf[col_name].dropna(),
        retained_pdf[col_name].dropna()
    )
    cm = churned_pdf[col_name].mean()
    rm = retained_pdf[col_name].mean()
    print(f"{col_name:<20} {cm:>12.2f} {rm:>12.2f} {p:>10.4f}  {'SIGNIFICANT ✓' if p<0.05 else 'not significant'}")
 
print("\nKey insight: Churned customers stay only ~18 months vs ~38 for retained")
print("Key insight: Churned customers pay ~$13 more per month")

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Churn Rate by Key Business Segments", fontsize=14, fontweight="bold")
 
# Chart 1: Contract type
c1 = pdf.groupby("Contract")["label"].mean().reset_index()
axes[0].bar(c1["Contract"], c1["label"],
            color=["#4CAF50","#FF9800","#F44336"], alpha=0.85, edgecolor="white")
axes[0].set_title("By Contract Type"); axes[0].set_ylabel("Churn rate")
axes[0].set_ylim(0, 0.6)
for i, v in enumerate(c1["label"]):
    axes[0].text(i, v+0.01, f"{v:.1%}", ha="center", fontsize=10, fontweight="bold")
 
# Chart 2: Internet Service
c2 = pdf.groupby("InternetService")["label"].mean().reset_index()
axes[1].bar(c2["InternetService"], c2["label"],
            color=["#2196F3","#9C27B0","#FF5722"], alpha=0.85, edgecolor="white")
axes[1].set_title("By Internet Service"); axes[1].set_ylabel("Churn rate")
axes[1].set_ylim(0, 0.6)
for i, v in enumerate(c2["label"]):
    axes[1].text(i, v+0.01, f"{v:.1%}", ha="center", fontsize=10, fontweight="bold")
 
# Chart 3: Tenure distribution
axes[2].hist(churned_pdf["tenure"], bins=25, alpha=0.6,
             color="#F44336", label="Churned", density=True)
axes[2].hist(retained_pdf["tenure"], bins=25, alpha=0.6,
             color="#4CAF50", label="Retained", density=True)
axes[2].set_title("Tenure Distribution"); axes[2].set_xlabel("Tenure (months)")
axes[2].legend()
 
plt.tight_layout()
plt.savefig("/Volumes/workspace/default/raw_datasets/eda_charts.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to volume.")

In [0]:
corr_cols   = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "label"]
corr_matrix = pdf[corr_cols].corr()
 
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0,
            linewidths=0.5, linecolor="white")
plt.title("Correlation Matrix — Numeric Features vs Churn Label")
plt.tight_layout()
plt.savefig("/Volumes/workspace/default/raw_datasets/correlation.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [0]:
print("\n" + "="*55)
print("  PHASE 2 COMPLETE — KEY FINDINGS SUMMARY")
print("="*55)
print("""
Categorical features significant (chi-square, p < 0.05):
  1. Contract       → month-to-month = 43% churn
  2. InternetService→ Fiber optic customers churn most
  3. PaymentMethod  → Electronic check = highest risk
  4. OnlineSecurity → No security = higher churn
  5. TechSupport    → No support = higher churn
 
Numerical features significant (t-test, p < 0.05):
  1. tenure         → churners stay only ~18 months avg
  2. MonthlyCharges → churners pay ~$13 more per month
  3. TotalCharges   → lower total = newer, higher-risk
 
""")

In [0]:
df.write.mode("overwrite").format("delta").save(
    "/Volumes/workspace/default/raw_datasets/feature_engineering"
)